# PSC Robustness Analysis

In [37]:
def default_params(): 
    return {
        'current_model': 'M1', 
        'quantization': 'none', #['none',"int4", "int8", "float32", "float16"]
        'dataset': {
            'path': '/workspaces/CodeSmells/semeru-datasets/code_smells/transformation',
            'transformation_list': ['curated','RenameVariable-1','RenameVariable-2','Add2Equal','SwitchEqualExp','InfixDividing', 'SwitchRelation'],
            'excluded_transformations' : ['RenameVariable-1','RenameVariable-2'],
            #'excluded_transformations' : [],
            'content_column': 'code',
            'sampling_size': 500,
        },
        'alignments_path': '/workspaces/CodeSmells/data/extension/transformation/alignments',
        'robustness_results_path' : '/workspaces/CodeSmells/notebooks/extension/robustness_transformations/06_robustness_analysis_results',
        'cache_dir': '/workspaces/CodeSmells/datax/hugging_face_cache',
    }
params = default_params()


#### Imports

In [2]:
import pandas as pd
import os
import time
import numpy as np
from datasets import load_dataset 
from statistics import mean, median
import json
import torch
import gc
import math
import scipy.stats as stats



In [3]:
import seaborn as sns; sns.set_theme()
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.backends.backend_pdf import PdfPages

In [4]:
from sklearn.preprocessing import StandardScaler
from scipy.stats import boxcox

#### Boostrapping

In [5]:
#| export
def bootstrapping( np_data, np_func, size ):
    """Create a bootstrap sample given data and a function
    For instance, a bootstrap sample of means, or mediands. 
    The bootstrap replicates are a long as the original size
    we can choose any observation more than once (resampling with replacement:np.random.choice)
    """
    
    #Cleaning NaNs
    #np_data_clean = np_data[ np.logical_not( np.isnan(np_data) ) ] 
    
    #The size of the bootstrap replicate is as big as size
    #Creating the boostrap replicates as long as the orignal data size
    #This strategy might work as imputation 
    bootstrap_repl = [ np_func( np.random.choice( np_data, size=len(np_data) ) ) for i in range( size ) ]
    
    #logging.info("Covariate: " + cov) #Empirical Mean
    #logging.info("Empirical Mean: " + str(np.mean(np_data_clean))) #Empirical Mean
    #logging.info("Bootstrapped Mean: " + str( np.mean(bootstrap_repl) ) ) #Bootstrapped Mean
    
    return np.array( bootstrap_repl )

#### Load Aggregates

In [6]:
def get_all_aggregates():
    aggregations = {}
    try:
        for transformation in params['dataset']['transformation_list']:
                transformation_df = pd.read_json(f"{params['alignments_path']}/{params['current_model']}_q_{params['quantization']}/{transformation}/aligned_smells.json")
                transformation_df['transformation'] = transformation
                aggregations[transformation] = transformation_df
    except Exception:
         None
    return aggregations


In [7]:
aggregations = get_all_aggregates()
aggregations['base'] = aggregations.pop('curated')
aggregations['base']['transformation'] = 'base'
aggregations.keys()

dict_keys(['RenameVariable-1', 'RenameVariable-2', 'Add2Equal', 'SwitchEqualExp', 'InfixDividing', 'SwitchRelation', 'base'])

In [8]:
aggregations['base']['s_msg_id'].unique()

array(['W0311', 'C0301', 'C0303', 'C0415', 'W0212', 'W0612', 'C0114',
       'W0613', 'C0116', 'C0304', 'C0305', 'C0123', 'C0103', 'W0622',
       'C0209', 'W0511', 'C0321', 'C0200', 'R1705', 'R0914', 'R1735',
       'R0912', 'W1514', 'C2801', 'R1710', 'C0325', 'W0621', 'R1720',
       'R1732', 'W0611', 'W0719', 'W0104', 'W0707', 'W0718', 'R0917',
       'R0913', 'W0102', 'W1309', 'W1406', 'C3001', 'W0601'], dtype=object)

#### Distribution comparisson between transformations

In [9]:
# 3. Define the logit transform
def logit_transform(x, eps=1e-9):
    x_clipped = np.clip(x, eps, 1 - eps)
    return np.log(x_clipped / (1 - x_clipped))

In [10]:
def logit_transform_psc(smell_msg_id, prob_column, decimal_spaces, aggregations):
    filtered_aggregations = {}
    for aggregation_name, aggregation_df in aggregations.items():
        filtered_df = aggregation_df[aggregation_df['s_msg_id'] == smell_msg_id].copy()

        # Apply logit transformation
        filtered_df[prob_column+'_trans'] = logit_transform(filtered_df[prob_column])

        ## Outlier removal after transformation
        # Calculate Q1 (25th percentile) and Q3 (75th percentile)
        Q1 = filtered_df[prob_column+'_trans'].quantile(0.25)
        Q3 = filtered_df[prob_column+'_trans'].quantile(0.75)

        # Compute the Interquartile Range (IQR)
        IQR = Q3 - Q1

        # Define lower and upper bounds for outliers
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        # Filter out the outliers
        filtered_df = filtered_df[(filtered_df[prob_column+'_trans'] >= lower_bound) & 
                                  (filtered_df[prob_column+'_trans'] <= upper_bound)]

        # Round to specified decimal places
        filtered_df[prob_column+'_trans'] = filtered_df[prob_column+'_trans'].round(decimal_spaces)

        filtered_aggregations[aggregation_name] = filtered_df
    return filtered_aggregations

In [11]:
def plot_density_distributions(smell_msg_id, psc_column, filtered_aggregations, ax, plot_stat= 'density'):
    combined_aggregates = pd.concat(filtered_aggregations.values(), ignore_index=True)
    # Plot histogram of actual data
    plt.figure(figsize=(10, 6))
    # Plot without the extra label parameter
    sns.histplot(x=psc_column, 
                  data=combined_aggregates, 
                  hue='transformation', 
                  kde=True, 
                  bins=500, 
                  stat=plot_stat, alpha= 0.3, ax=ax)


In [12]:
def compute_cliff_delta(distribution_1, distribution_2):
    n1, n2 = len(distribution_1), len(distribution_2)
    greater = sum(x > y for x in distribution_1 for y in distribution_2)
    lesser = sum(x < y for x in distribution_1 for y in distribution_2)
    return (greater - lesser) / (n1 * n2)

In [13]:
def bootstrap_ks(distribution_1, distribution_2, n_boot=1000):
    n1, n2 = len(distribution_1), len(distribution_2)
    D_boot = [stats.ks_2samp(np.random.choice(distribution_1, size=n1, replace=True),
                        np.random.choice(distribution_2, size=n2, replace=True))[0] 
              for _ in range(n_boot)]
    return np.percentile(D_boot, [2.5, 97.5])

In [14]:
def compare_distributions(distribution_1, distribution_2):
    np.random.seed(0)

    # Make distributions the same size
    min_size = min(len(distribution_1), len(distribution_2))
    if min_size <= 0: return None
    distribution_1 = np.random.choice(distribution_1, size=min_size, replace=False)
    distribution_2 = np.random.choice(distribution_2, size=min_size, replace=False)

    # Determine n_boot dynamically
    n_boot = max(100, min(1000, min_size // 10))
    
    ks_statistic, p_value = stats.ks_2samp(distribution_1, distribution_2, alternative='two-sided')
    st_effect_size = ks_statistic * math.sqrt((min_size * min_size) / (min_size + min_size))
    cliff_delta = compute_cliff_delta(distribution_1, distribution_2)
    wasserstein_dist = stats.wasserstein_distance(distribution_1, distribution_2)
    ci_ks_lower, ci_ks_upper = bootstrap_ks(distribution_1, distribution_2, n_boot)

    # Final decision based on all computed values
    practically_equivalent = (ci_ks_upper < 0.21) and (abs(cliff_delta) < 0.147) and (p_value > 0.05) and (wasserstein_dist < 0.1)
    
    return {
        "ks_stat": ks_statistic,
        "ci_ks_stat": (ci_ks_lower, ci_ks_upper),
        "p_value": p_value,
        "st_effect_size": st_effect_size,
        "cliffs_delta": cliff_delta,
        "wasserstein_dist" : wasserstein_dist,
        "equivalence": practically_equivalent
    }

In [15]:
def get_unique_smell_ids(aggregations): 
    # Get all unique smell_msg_id values from your aggregations dictionary
    unique_smell_ids = set()
    for aggregation_df in aggregations.values():
        unique_smell_ids.update(aggregation_df['s_msg_id'].unique())
    unique_smell_ids = list(unique_smell_ids)
    return unique_smell_ids

In [16]:
def plot_transformation_distributions(prob_column, aggregations):
    # Get all unique smell_msg_id values from your aggregations dictionary
    unique_smell_ids = get_unique_smell_ids(aggregations)

    # Determine grid dimensions (using a square-like layout)
    n_plots = len(unique_smell_ids)
    ncols = math.ceil(math.sqrt(n_plots))
    nrows = math.ceil(n_plots / ncols)

    # Create the figure and axes using constrained_layout to automatically adjust spacing
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5, nrows * 4), constrained_layout=True)
    axes = axes.flatten()  # Flatten for easier iteration


    # Loop over each smell_msg_id and plot in its corresponding subplot
    for i, smell_msg_id in enumerate(unique_smell_ids):
        # Transform data for current smell_msg_id
        filtered_aggregations = logit_transform_psc(smell_msg_id, prob_column, 4, aggregations)
        plot_density_distributions(smell_msg_id, prob_column+'_trans', filtered_aggregations, axes[i], 'density')
        # Set titles and labels for this subplot
        axes[i].set_title(f"Distributions of {smell_msg_id} by Transformation")
        axes[i].set_xlabel("PSC (Propensity Smelly Score)")

    # Remove any unused subplots if the grid is larger than needed
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    fig.savefig(f"{params['robustness_results_path']}/{prob_column}.pdf", format="pdf")
    fig.savefig(f"{params['robustness_results_path']}/{prob_column}.png", format="png", dpi=300)
    
    # Close the figure to avoid displaying it
    plt.close(fig)


In [17]:
def perform_base_stat_tests(psc_column, aggregations):
    stat_test_df = pd.DataFrame(columns=['s_msg_id', 'transformation', 'ks_stat', 'ci_ks_stat', 'p_value', 'st_effect_size', 'cliffs_delta', 'wasserstein_dist', 'equivalence'])
    for code_smell_id in get_unique_smell_ids(aggregations):
        for transformation in [agg for agg in list(aggregations.keys()) if agg != 'base']:
            filtered_aggregations = logit_transform_psc(code_smell_id, psc_column, 4, aggregations)
            stat_test_results = compare_distributions(filtered_aggregations['base'][psc_column+'_trans'], filtered_aggregations[transformation][psc_column+'_trans'])
            if stat_test_results is None: continue
            stat_test_results['transformation'] = transformation
            stat_test_results['s_msg_id'] = code_smell_id
            stat_test_df.loc[len(stat_test_df)] = stat_test_results
    return stat_test_df

In [18]:
def perform_anova(distribution_list):
    # Ensure input is a list of at least two distributions
    if len(distribution_list) < 2:
        raise ValueError("At least two distributions are required for ANOVA.")
    
    # Ensure all distributions have the same size
    min_size = min(len(distribution) for distribution in distribution_list)
    if min_size <= 0: return None
    distribution_list = [distribution[:min_size] for distribution in distribution_list]
    
    # Perform ANOVA test
    f_statistic, p_value = stats.f_oneway(*distribution_list)
    
    # Compute effect size (eta squared)
    mean_overall = np.mean(np.concatenate(distribution_list))
    
    sum_squares_between = sum(len(distribution) * (np.mean(distribution) - mean_overall) ** 2 for distribution in distribution_list)
    sum_squares_total = sum((sample - mean_overall) ** 2 for distribution in distribution_list for sample in distribution)
    eta_squared = sum_squares_between / sum_squares_total if sum_squares_total != 0 else 0
    
    return {
        'f_statistic': f_statistic,
        'p_value': p_value,
        'eta_squared': eta_squared
    }

In [19]:
def perform_anova_tests(psc_column, aggregations):
    stat_test_df = pd.DataFrame(columns=['s_msg_id', 'f_statistic', 'p_value', 'eta_squared'])
    for code_smell_id in get_unique_smell_ids(aggregations):
        filtered_aggregations = logit_transform_psc(code_smell_id, psc_column, 4, aggregations)
        stat_test_results = perform_anova([aggregation_df[psc_column+'_trans'] for aggregation_df in filtered_aggregations.values()])
        if stat_test_results is None: continue
        stat_test_results['s_msg_id'] = code_smell_id
        stat_test_df.loc[len(stat_test_df)] = stat_test_results
    return stat_test_df
    

#### Comparisons against base

In [20]:
aggregations['base'].columns

Index(['id', 'commit_id', 'repo', 'path', 'file_name', 'fun_name',
       'commit_message', 'code', 'url', 'language', 'ast_errors',
       'n_ast_errors', 'ast_levels', 'n_whitespaces', 'n_words', 'vocab_size',
       'complexity', 'nloc', 'token_counts', 'n_ast_nodes', 'n_identifiers',
       's_msg_id', 's_line', 's_column', 's_end_line', 's_end_column',
       's_code', 'category', 'input_lenght', 'input_ids', 'max_prob',
       'min_prob', 'actual_prob', 'loss', 'code_smell_pos',
       'code_smell_actual_logits', 'code_smell_max_logits',
       'code_smell_min_logits', 'code_smell_actual_prob_median',
       'code_smell_max_prob_median', 'code_smell_min_prob_median',
       'code_smell_actual_prob_mean', 'code_smell_max_prob_mean',
       'code_smell_min_prob_mean', 'code_smell_psc_entropy',
       'code_smell_psc_relative', 'transformation'],
      dtype='object')

In [21]:
psc_column = 'code_smell_psc_relative'
stat_tests_df = perform_base_stat_tests(psc_column, aggregations)

/usr/local/lib/python3.11/dist-packages/scipy/stats/_axis_nan_policy.py:579: RuntimeWarning: ks_2samp: Exact calculation unsuccessful. Switching to method=asymp.
  res = hypotest_fun_out(*samples, **kwds)
/usr/local/lib/python3.11/dist-packages/scipy/stats/_axis_nan_policy.py:579: RuntimeWarning: ks_2samp: Exact calculation unsuccessful. Switching to method=asymp.
  res = hypotest_fun_out(*samples, **kwds)
/usr/local/lib/python3.11/dist-packages/scipy/stats/_axis_nan_policy.py:579: RuntimeWarning: ks_2samp: Exact calculation unsuccessful. Switching to method=asymp.
  res = hypotest_fun_out(*samples, **kwds)
/usr/local/lib/python3.11/dist-packages/scipy/stats/_axis_nan_policy.py:579: RuntimeWarning: ks_2samp: Exact calculation unsuccessful. Switching to method=asymp.
  res = hypotest_fun_out(*samples, **kwds)
/usr/local/lib/python3.11/dist-packages/scipy/stats/_axis_nan_policy.py:579: RuntimeWarning: ks_2samp: Exact calculation unsuccessful. Switching to method=asymp.
  res = hypotest_f

In [22]:
len(stat_tests_df[stat_tests_df['equivalence']==True])

145

In [23]:
stat_tests_df[(stat_tests_df['equivalence']==True) & (stat_tests_df['transformation']=='SwitchEqualExp')]

,s_msg_id,transformation,ks_stat,ci_ks_stat,p_value,st_effect_size,cliffs_delta,wasserstein_dist,equivalence
3,W1309,SwitchEqualExp,0.019802,"(0.038304455445544554, 0.09783415841584156)",0.999999,0.281439,0.002285,0.025935,True
9,R0914,SwitchEqualExp,0.000000,"(0.029925187032418952, 0.09239401496259347)",1.000000,0.000000,0.000000,0.000000,True
15,C0305,SwitchEqualExp,0.004988,"(0.0036783042394014963, 0.04987531172069826)",1.000000,0.070622,-0.001723,0.000672,True
20,C0200,SwitchEqualExp,0.030488,"(0.03348577235772358, 0.1016260162601626)",0.976443,0.478183,0.018970,0.029259,True
26,W0621,SwitchEqualExp,0.011204,"(0.03361344537815126, 0.11778711484593833)",1.000000,0.149696,0.006308,0.022895,True
32,W0511,SwitchEqualExp,0.025229,"(0.034403669724770644, 0.11129587155963301)",0.999101,0.372507,-0.023251,0.033066,True
38,R0917,SwitchEqualExp,0.000000,"(0.030878859857482184, 0.09875296912114008)",1.000000,0.000000,0.000000,0.000000,True
44,R0913,SwitchEqualExp,0.000000,"(0.02798780487804878, 0.1048780487804878)",1.000000,0.000000,0.000000,0.000000,True
50,C0325,SwitchEqualExp,0.029613,"(0.03189066059225513, 0.10153758542141225)",0.990739,0.438729,0.023609,0.036487,True
60,R1732,SwitchEqualExp,0.017032,"(0.0340632603406326, 0.09245742092457421)",1.000000,0.244153,-0.013551,0.025153,True


#### Comparisons all included

In [24]:
aggregations['base'].columns

Index(['id', 'commit_id', 'repo', 'path', 'file_name', 'fun_name',
       'commit_message', 'code', 'url', 'language', 'ast_errors',
       'n_ast_errors', 'ast_levels', 'n_whitespaces', 'n_words', 'vocab_size',
       'complexity', 'nloc', 'token_counts', 'n_ast_nodes', 'n_identifiers',
       's_msg_id', 's_line', 's_column', 's_end_line', 's_end_column',
       's_code', 'category', 'input_lenght', 'input_ids', 'max_prob',
       'min_prob', 'actual_prob', 'loss', 'code_smell_pos',
       'code_smell_actual_logits', 'code_smell_max_logits',
       'code_smell_min_logits', 'code_smell_actual_prob_median',
       'code_smell_max_prob_median', 'code_smell_min_prob_median',
       'code_smell_actual_prob_mean', 'code_smell_max_prob_mean',
       'code_smell_min_prob_mean', 'code_smell_psc_entropy',
       'code_smell_psc_relative', 'transformation'],
      dtype='object')

In [35]:
filtered_transformations['RenameVariable-1'].columns

Index(['id', 'commit_id', 'repo', 'path', 'file_name', 'commit_message', 'url',
       'language', 'category', 'code', 's_msg_id', 's_line', 's_column',
       's_end_line', 's_end_column', 's_code', 'n_whitespaces', 'n_words',
       'vocab_size', 'fun_name', 'complexity', 'nloc', 'token_counts',
       'ast_errors', 'ast_levels', 'n_ast_nodes', 'n_ast_errors',
       'n_identifiers', 'input_ids', 'input_lenght', 'max_prob', 'min_prob',
       'actual_prob', 'loss', 'code_smell_pos', 'code_smell_actual_logits',
       'code_smell_max_logits', 'code_smell_min_logits',
       'code_smell_actual_prob_median', 'code_smell_max_prob_median',
       'code_smell_min_prob_median', 'code_smell_actual_prob_mean',
       'code_smell_max_prob_mean', 'code_smell_min_prob_mean',
       'code_smell_psc_entropy', 'code_smell_psc_relative', 'transformation'],
      dtype='object')

In [38]:
psc_column = 'code_smell_actual_prob_median'
filtered_transformations = {k: v for k, v in aggregations.items() if k not in params['dataset']['excluded_transformations']}
anova_results = perform_anova_tests(psc_column, filtered_transformations)


anova_results['f_statistic'] = anova_results['f_statistic'].round(5)

anova_results['p_value'] = anova_results['p_value'].round(5)
anova_results['eta_squared'] = anova_results['eta_squared'].round(5)


anova_results = anova_results.sort_values(by=['f_statistic'], ascending=True)
anova_results

,s_msg_id,f_statistic,p_value,eta_squared
0,R0914,-0.00000,NaN,0.00000
31,R1710,-0.00000,NaN,0.00000
26,C0114,0.00000,1.00000,0.00000
39,W0102,0.00000,1.00000,0.00000
11,R0912,0.00000,1.00000,0.00000
7,R0913,0.00000,1.00000,0.00000
20,C0116,0.00000,1.00000,0.00000
6,R0917,0.00000,1.00000,0.00000
15,W0718,0.00177,0.99999,0.00000
22,C0415,0.00253,0.99999,0.00000


In [ ]:
# Compute summary statistics (mean, std, 95% CI) per s_msg_id based on PSC values (original scale)
summary_rows = []
for code_smell_id in get_unique_smell_ids(filtered_transformations):
    # Get raw (non-logit) aggregations for this smell id by subsetting each transformation
    raw_aggs = {}
    for name, df in filtered_transformations.items():
        raw_aggs[name] = df[df['s_msg_id'] == code_smell_id].copy()

    # Concatenate all transformations for this smell id on the original PSC scale
    combined = pd.concat(raw_aggs.values(), ignore_index=True)
    col = psc_column
    values = combined[col].dropna().values
    if len(values) == 0:
        continue
    mean_val = np.mean(values)
    std_val = np.std(values, ddof=1) if len(values) > 1 else 0.0
    n = len(values)
    # 95% CI for the mean using normal approximation on original scale
    stderr = std_val / np.sqrt(n) if n > 1 else 0.0
    ci_half_width = 1.96 * stderr
    # Single column with CI expressed as mean ± 1.96 * stderr (original PSC scale)
    ci_95 = f"{mean_val:.5f} ± {ci_half_width:.5f}"

    summary_rows.append({
        's_msg_id': code_smell_id,
        'n': n,
        'mean_psc': mean_val,
        'std_psc': std_val,
        'stderr_psc': stderr,
        'ci95_mean_pm_1_96_se_psc': ci_95,
    })

anova_summary = pd.DataFrame(summary_rows)
anova_summary

,s_msg_id,n,mean_trans_psc,std_trans_psc,stderr_trans_psc,ci95_mean_pm_1_96_se
0,R0914,2500,-4.843198,1.954352,0.039087,-4.84320 ± 0.07661
1,W1309,2498,1.548738,2.633287,0.052687,1.54874 ± 0.10327
2,C0305,2397,3.904921,2.137537,0.043660,3.90492 ± 0.08557
3,C0200,2500,5.010480,2.193732,0.043875,5.01048 ± 0.08599
4,W0621,2410,-0.879890,2.836470,0.057779,-0.87989 ± 0.11325
5,W0511,2400,-0.775663,2.509701,0.051229,-0.77566 ± 0.10041
6,R0917,2500,-3.915241,2.668139,0.053363,-3.91524 ± 0.10459
7,R0913,2500,-4.138339,2.647504,0.052950,-4.13834 ± 0.10378
8,C0325,2480,1.973959,2.326316,0.046714,1.97396 ± 0.09156
9,W1514,2494,2.399662,2.699867,0.054062,2.39966 ± 0.10596


### Vissual Comparison 

In [27]:
psc_column = 'code_smell_psc_relative'
filtered_transformations = {k: v for k, v in aggregations.items() if k not in params['dataset']['excluded_transformations']}
#plot_transformation_distributions(psc_column, filtered_transformations)

### Variability

In [28]:
## variability (std)
alignments_df = aggregations['Add2Equal']
grouped = alignments_df.groupby('s_msg_id')['code_smell_psc_relative'].describe()
mean_of_means = grouped['mean'].mean()
print("Mean of means:", mean_of_means)

Mean of means: 0.6546369388304284
